# Task 3 — The Smoking Gun (interpretability)

*Why* does the detector cry "AI"? Four experiments, all on the **canonical RoBERTa model** from Task 2 (`models/tierC_roberta_tertiary`):
1. **Saliency** (Captum LIG) on styled-AI imposters and on the SOP.
2. **AI-ism aggregation** — ranked AI-signaling tokens, separately for neutral vs styled. Tests the folklore ("delve/tapestry") against the reality (register/rhythm).
3. **Proper-noun ablation** — mask PERSON/GPE/ORG, re-measure accuracy. The delta quantifies name-leak.
4. **SOP disentangling** — is the SOP flagged AI for modern *register* or for generative *fingerprints*?

> NOTE: set `MODEL_NAME`/`ADAPTER` below to match whichever model you made canonical. RoBERTa is the default because it generalised better to the modern-human probe (86.7% vs 70%).

In [1]:
!pip install -q captum transformers peft spacy scikit-learn
!python -m spacy download en_core_web_sm -q

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [ ]:
import json, numpy as np, pandas as pd, torch
from pathlib import Path
from collections import defaultdict
import spacy
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel
from captum.attr import LayerIntegratedGradients


MODEL_NAME = "roberta-base"
ADAPTER    = "models/tierC_roberta_tertiary"
SPECIAL    = {"[CLS]","[SEP]","[PAD]","<s>","</s>","<pad>"}

DATA = Path("data/dataset")
device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
base = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
model = PeftModel.from_pretrained(base, ADAPTER).to(device).eval()
nlp = spacy.load("en_core_web_sm", disable=["lemmatizer"])

def load_jsonl(p): return [json.loads(l) for l in Path(p).read_text(encoding="utf-8").splitlines() if l.strip()]
test = pd.DataFrame(load_jsonl(DATA/"test.jsonl"))
print("test paragraphs:", len(test), "| classes:", test["label"].value_counts().to_dict())

c:\Users\Richa\OneDrive\Documents\Projects\precog-task\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 4288.79it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if y

test paragraphs: 189 | classes: {'human': 63, 'ai_neutral': 63, 'ai_styled': 63}


## 1. Saliency: Captum Layer Integrated Gradients
Attribute the predicted-class logit back to the input tokens. We read the *pattern* (register vs LLM-cliche), not individual function words, which get high attribution simply by being frequent.

In [ ]:
def predict_logits(input_ids, attention_mask):
    return model(input_ids=input_ids, attention_mask=attention_mask).logits

lig = LayerIntegratedGradients(predict_logits, model.get_input_embeddings())

def clean_tok(t):
    # RoBERTa BPE: 'Ġ' marks a leading space, 'Ċ' is a newline.
    return t.replace("\u0120", "").replace("\u010a", "").strip()

def attributions(text, target=None):
    enc = tok(text, return_tensors="pt", truncation=True, max_length=256).to(device)
    if target is None:
        target = int(model(**enc).logits.argmax(1))
    ref = torch.full_like(enc["input_ids"], tok.pad_token_id)
    atts = lig.attribute(inputs=enc["input_ids"], baselines=ref,
                         additional_forward_args=(enc["attention_mask"],),
                         target=target, n_steps=32)
    scores = atts.sum(-1).squeeze(0).detach().cpu().numpy()
    toks = tok.convert_ids_to_tokens(enc["input_ids"].squeeze(0))
    return list(zip(toks, scores)), target

def top_tokens(text, k=12, target=None):
    pairs, tgt = attributions(text, target)
    clean = [(clean_tok(t), s) for t, s in pairs]
    clean = [(t, s) for t, s in clean if t and t not in SPECIAL]
    top = sorted(clean, key=lambda x: -x[1])[:k]
    print(f"predicted class {tgt} (0=human 1=neutral 2=styled)")
    print("top tokens:", [t for t, _ in top])
    return top

In [ ]:
for txt in test[test.label == "ai_styled"]["text"].head(2):
    print("="*60); print(txt[:140], "..."); top_tokens(txt)

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


It was an era of ardent affection, it was an era of callous indifference, it was the season of unbridled happiness, it was the season of pro ...
predicted class 2 (0=human 1=neutral 2=styled)
top tokens: ['.', '.', '.', 'season', 'season', 'un', 'ourselves', 'ous', 'affection', 'love', 'longing', 'measure']
It was a season of sorrow, it was a season of longing, it was the age of departure, it was the age of nostalgia. It is a poignant truth that ...
predicted class 2 (0=human 1=neutral 2=styled)
top tokens: ['.', 'truth', '.', 'of', '.', 'It', 'season', 'season', 'that', 'of', 'longing', 'hip']


## 2. AI-ism aggregation (neutral vs styled, separately)
Aggregate token attributions toward each AI class across many paragraphs, to rank the words that most signal "this is AI".

In [5]:
def rank_ai_words(label, target_class, n_docs=120, min_count=3, k=25):
    agg = defaultdict(list)
    for txt in test[test.label == label]["text"].head(n_docs):
        pairs, _ = attributions(txt, target=target_class)
        for t, s in pairs:
            ct = clean_tok(t)
            if ct.isalpha() and ct not in SPECIAL:
                agg[ct].append(float(s))
    ranked = sorted(((w, float(np.mean(v)), len(v)) for w, v in agg.items() if len(v) >= min_count),
                    key=lambda x: -x[1])
    print(f"\nTop AI-signaling tokens for {label} (target class {target_class}):")
    for w, m, c in ranked[:k]:
        print(f"  {w:15s} {m:+.4f}  (n={c})")
    pd.DataFrame(ranked, columns=["word","mean_attr","count"]).to_csv(f"output/task-3/results_ai_words_{label}.csv", index=False)
    return ranked

_ = rank_ai_words("ai_neutral", 1)
_ = rank_ai_words("ai_styled", 2)


Top AI-signaling tokens for ai_neutral (target class 1):
  At              +0.6209  (n=3)
  Domestic        +0.4168  (n=5)
  strength        +0.3880  (n=3)
  shaping         +0.3170  (n=12)
  settle          +0.2927  (n=3)
  Disapp          +0.2170  (n=3)
  transition      +0.2081  (n=7)
  space           +0.2063  (n=4)
  shift           +0.2035  (n=5)
  approach        +0.1985  (n=4)
  key             +0.1981  (n=4)
  constructive    +0.1921  (n=3)
  It              +0.1731  (n=5)
  working         +0.1730  (n=4)
  environment     +0.1723  (n=20)
  caregivers      +0.1719  (n=3)
  environments    +0.1602  (n=6)
  boost           +0.1588  (n=5)
  contribute      +0.1564  (n=3)
  strategies      +0.1562  (n=7)
  framework       +0.1558  (n=3)
  negative        +0.1514  (n=8)
  play            +0.1504  (n=4)
  communicate     +0.1501  (n=3)
  built           +0.1396  (n=6)

Top AI-signaling tokens for ai_styled (target class 2):
  Yet             +0.1917  (n=4)
  surroundings    +0.1308

## 3. Proper-noun ablation (The differentiator)
Mask PERSON/GPE/ORG/LOC/FAC with spaCy NER and re-measure accuracy.

In [6]:
def mask_entities(text):
    doc = nlp(text)
    spans = [(e.start_char, e.end_char, e.label_) for e in doc.ents
             if e.label_ in ("PERSON","GPE","ORG","LOC","FAC")]
    repl = {"PERSON":"[PERSON]","GPE":"[PLACE]","LOC":"[PLACE]","FAC":"[PLACE]","ORG":"[ORG]"}
    for s, e, lab in sorted(spans, key=lambda x: -x[0]):   # right-to-left keeps indices valid
        text = text[:s] + repl[lab] + text[e:]
    return text

def predict_df(df):
    preds = []
    for _, row in df.iterrows():
        enc = tok(row["text"], return_tensors="pt", truncation=True, max_length=256).to(device)
        with torch.no_grad():
            preds.append(int(model(**enc).logits.argmax(1)))
    return np.array(preds)

raw_pred = predict_df(test)
masked = test.copy(); masked["text"] = masked["text"].map(mask_entities)
msk_pred = predict_df(masked)

y = test["class_id"].values
acc_raw = (raw_pred == y).mean(); acc_msk = (msk_pred == y).mean()
print(f"OVERALL  raw={acc_raw:.3f}  masked={acc_msk:.3f}  delta={acc_raw-acc_msk:+.3f}")
print("\nper-class accuracy (raw -> masked):")
for cid, lab in [(0,"human"),(1,"ai_neutral"),(2,"ai_styled")]:
    m = y == cid
    print(f"  {lab:11s} {(raw_pred[m]==cid).mean():.3f} -> {(msk_pred[m]==cid).mean():.3f}")

OVERALL  raw=1.000  masked=1.000  delta=+0.000

per-class accuracy (raw -> masked):
  human       1.000 -> 1.000
  ai_neutral  1.000 -> 1.000
  ai_styled   1.000 -> 1.000


## 4. SOP disentangling (Register leak or real AI fingerprint?)
The SOP is AI-assisted **and** modern-register, so its "AI" label is entangled.

In [7]:
sop = load_jsonl(Path("data/dataset")/"probe_sop.jsonl")
LLM_TELLS = {"delve","leverage","underscore","tapestry","crucial","pivotal","realm",
             "intricate","multifaceted","nuanced","robust","holistic","seamless"}
hits = []
for r in sop[:5]:
    print("="*60); print(r["text"][:130], "...")
    top = top_tokens(r["text"])
    hits += [t for t, _ in top if t.lower() in LLM_TELLS]
print("\nLLM-cliche tokens found among top-saliency SOP tokens:", hits or "NONE")
print("=> if NONE/few, the SOP is flagged mainly for modern REGISTER, not generative fingerprints.")

There are some labs you discover and think, “this is interesting.” And then there are a few rare
ones that make you feel, almost i ...
predicted class 2 (0=human 1=neutral 2=styled)
top tokens: ['.', '.', 'people', 'What', 's', 'rel', '.', '.', '.', 'ones', 'not', 'Ŀ']
Over time, I have realized that this is the kind of researcher I want to become. I enjoy building
AI systems, but even more than t ...
predicted class 1 (0=human 1=neutral 2=styled)
top tokens: ['AI', 'AI', 'systems', 'researcher', 'they', 'want', 'perform', 'building', 'only', 'measured', 'want', 'models']
A major reason for this excitement is Prof. Ponnurangam Kumaraguru. From everything I have
seen, PK sir feels like a larger-than-l ...
predicted class 1 (0=human 1=neutral 2=styled)
top tokens: ['.', 'research', 'career', 'meaningful', 'systems', 'reson', 'with', 'major', 'ates', 'want', 'especially', 'AI']
My own journey into AI has been driven by a mix of curiosity, experimentation, and a very
strong desire to build

## 5. Error analysis

In [ ]:
test_r = test.reset_index(drop=True)
errs = []
for pos, row in test_r.iterrows():
    if raw_pred[pos] != row["class_id"]:
        errs.append({"true": row["label"], "pred": int(raw_pred[pos]),
                     "text": row["text"][:200]})
err_df = pd.DataFrame(errs)
print(f"{len(err_df)} misclassified of {len(test_r)}")
print(err_df.head(6).to_string() if len(err_df) else "no errors (in-distribution test is perfectly separable)")
err_df.to_csv("output/task-3/results_errors.csv", index=False)

0 misclassified of 189
no errors (in-distribution test is perfectly separable)


## 6. Genre probe
Mean probability mass on the human class for prose the model never trained on.

In [9]:
def p_human(text):
    enc = tok(text, return_tensors="pt", truncation=True, max_length=256).to(device)
    with torch.no_grad():
        return float(torch.softmax(model(**enc).logits, 1).squeeze(0)[0])

for probe in ["probe_modern_human", "probe_sop"]:
    rows = load_jsonl(Path("data/dataset")/f"{probe}.jsonl")
    ph = np.mean([p_human(r["text"]) for r in rows])
    print(f"{probe}: mean P(human) = {ph:.3f}  (n={len(rows)})")

probe_modern_human: mean P(human) = 0.825  (n=60)
probe_sop: mean P(human) = 0.000  (n=11)
